In [0]:
# Step 1: Read JSON file with multiline option
path = "/Volumes/dev_account/stage/account/raw/customer/inbound/Nested_json_data_corrected.jsonl"

print("📁 Reading JSON file...")
print(f"   Path: {path}\n")

# multiline=true: Reads entire JSON as one record (for pretty-printed JSON)
df = spark.read.option("multiline", "true").json(path)

display(df)

## Understanding JSON Data Types in Spark

When Spark reads JSON, it infers the schema with these data types:

### 1. **Primitive Types**
* `string` - Text data
* `long` / `integer` - Numbers
* `double` - Decimal numbers
* `boolean` - true/false

### 2. **Complex Types**
* **`array`** - Collection of elements (like a list)
  - Example: `["item1", "item2", "item3"]`
  - Creates multiple rows when you **EXPLODE**
  
* **`struct`** - Nested object (like a dictionary)
  - Example: `{"city": "New York", "state": "NY"}`
  - Access fields with **dot notation** or **FLATTEN**

### Key Difference:
* **Array** = Multiple values → Use **EXPLODE** to create rows
* **Struct** = Nested fields → Use **dot notation** or **FLATTEN** to access

In [0]:
# Step 2: Display the nested JSON data as-is
print("📄 Raw Nested JSON Data:")
print("   Notice: Data is deeply nested with arrays and structs\n")

display(df)

print("\n👉 Observation:")
print("   - You see complex nested structures")
print("   - Arrays shown as [...] ")
print("   - Structs shown as {field1: value1, field2: value2}")
print("   - Hard to query and analyze in this format!")

## Understanding the JSON Structure

Our JSON has this hierarchy:

```
entities (ARRAY) ← Top-level array
 └─ element (STRUCT)
     ├─ entityId
     ├─ entityType
     ├─ data (STRUCT) ← Nested struct
     │   └─ attributes (ARRAY) ← Nested array
     │       └─ element (STRUCT)
     │           ├─ attributeId
     │           ├─ name
     │           └─ property (STRUCT) ← Deeply nested
     │               ├─ firstName
     │               ├─ lastName
     │               ├─ contact (STRUCT)
     │               └─ addressDetails (STRUCT)
     └─ relationships (ARRAY)
         └─ element (STRUCT)
             └─ targetEntity (STRUCT)
```

**Challenge:** How do we work with this nested data?

**Solutions:**
1. **EXPLODE** - Convert arrays to separate rows
2. **FLATTEN** - Access nested struct fields with dot notation

## Step 3: What is EXPLODE? 💥

**EXPLODE** converts an **array** into multiple **rows**.

### Before EXPLODE:
```
| id | items              |
|----|--------------------|
| 1  | ["A", "B", "C"]   |
```

### After EXPLODE:
```
| id | item |
|----|------|
| 1  | A    |
| 1  | B    |
| 1  | C    |
```

### In PySpark:
```python
from pyspark.sql.functions import explode

df_exploded = df.select(explode("array_column").alias("item"))
```

**Use Case:** When you have arrays and want each element as a separate row.

In [0]:
# Step 3A: EXPLODE the entities array
from pyspark.sql.functions import explode, col

print("💥 EXPLODE: Converting entities array to rows\n")

# Before explode: 1 row with array of entities
print(f"Before EXPLODE: {df.count()} row(s)")

# After explode: Each entity becomes a separate row
df_exploded = df.select(explode("entities").alias("entity"))

print(f"After EXPLODE:  {df_exploded.count()} row(s)")
print("\n📊 New Schema:")
df_exploded.printSchema()

print("\n📄 Exploded Data (each entity is now a row):")
display(df_exploded)

## Step 4: What is FLATTEN? 🔽

**FLATTEN** means accessing nested **struct** fields using **dot notation**.

### Before FLATTEN:
```
| id | address                                  |
|----|------------------------------------------|
| 1  | {city: "NYC", state: "NY", zip: 10001} |
```

### After FLATTEN:
```
| id | city | state | zip   |
|----|------|-------|-------|
| 1  | NYC  | NY    | 10001 |
```

### In PySpark:
```python
# Method 1: Dot notation
df.select("id", "address.city", "address.state")

# Method 2: Using col()
df.select(col("address.city"), col("address.state"))
```

**Use Case:** When you have nested structs and want to promote fields to top-level columns.

In [0]:
# Step 4A: FLATTEN nested structs using dot notation
print("🔽 FLATTEN: Accessing nested struct fields\n")

# Access top-level fields from entity struct
df_flattened = df_exploded.select(
    col("entity.entityId").alias("entityId"),
    col("entity.entityType").alias("entityType"),
    col("entity.data").alias("data"),
    col("entity.relationships").alias("relationships")
)

print("📊 Schema after flattening entity:")
df_flattened.printSchema()

print("\n📄 Flattened Data:")
display(df_flattened.limit(5))

print("\n👉 Notice: We brought entityId and entityType to top level!")

## Step 5: Combine EXPLODE + FLATTEN 🎯

For complex nested JSON, you often need to:
1. **EXPLODE** arrays to create rows
2. **FLATTEN** structs to create columns
3. Repeat for multiple levels of nesting

### Strategy:
```
Original Data
    ↓
EXPLODE array level 1
    ↓
FLATTEN struct fields
    ↓
EXPLODE nested array level 2
    ↓
FLATTEN nested struct fields
    ↓
Final flat table
```

In [0]:
# Step 5A: Complete example - Extract person information
print("🎯 COMPLETE EXAMPLE: Extract customer data\n")

# Step 1: Explode entities array
df_step1 = df.select(explode("entities").alias("entity"))

# Step 2: Flatten to get entityId, entityType, and attributes array
df_step2 = df_step1.select(
    col("entity.entityId").alias("entityId"),
    col("entity.entityType").alias("entityType"),
    col("entity.data.attributes").alias("attributes")
)

# Step 3: Explode attributes array
df_step3 = df_step2.select(
    "entityId",
    "entityType",
    explode("attributes").alias("attribute")
)

# Step 4: Flatten attribute properties to get person details
df_final = df_step3.select(
    "entityId",
    "entityType",
    col("attribute.attributeId").alias("attributeId"),
    col("attribute.name").alias("attributeName"),
    col("attribute.property.firstName").alias("firstName"),
    col("attribute.property.lastName").alias("lastName"),
    col("attribute.property.age").alias("age"),
    col("attribute.property.contact.email").alias("email"),
    col("attribute.property.contact.phone").alias("phone"),
    col("attribute.property.addressDetails.street").alias("street"),
    col("attribute.property.addressDetails.city").alias("city"),
    col("attribute.property.addressDetails.state").alias("state"),
    col("attribute.property.addressDetails.postalCode").alias("postalCode")
)

print("✅ Transformation complete!")
print(f"📊 Rows: {df_final.count()}")
print("\n📊 Final Flattened Schema:")
df_final.printSchema()

print("\n📄 Final Flattened Data:")
display(df_final)

In [0]:
# Step 6: Extract account information from relationships
print("💳 Extract Account Information from Relationships\n")

# Explode entities -> relationships -> targetEntity -> attributes
df_accounts = df.select(explode("entities").alias("entity")) \
    .select(
        col("entity.entityId").alias("personEntityId"),
        explode("entity.relationships").alias("relationship")
    ) \
    .select(
        "personEntityId",
        col("relationship.relationshipType").alias("relationshipType"),
        col("relationship.targetEntity.entityId").alias("accountEntityId"),
        col("relationship.targetEntity.entityType").alias("accountEntityType"),
        explode("relationship.targetEntity.data.attributes").alias("account_attr")
    ) \
    .select(
        "personEntityId",
        "relationshipType",
        "accountEntityId",
        col("account_attr.property.accountNumber").alias("accountNumber"),
        col("account_attr.property.accountType").alias("accountType"),
        col("account_attr.property.balance").alias("balance"),
        col("account_attr.property.currency").alias("currency"),
        col("account_attr.property.active").alias("active")
    )

print("✅ Account extraction complete!")
print(f"📊 Rows: {df_accounts.count()}")
print("\n📊 Schema:")
df_accounts.printSchema()

print("\n📄 Account Data:")
display(df_accounts)

## 📚 Summary: Working with Nested JSON

### Key Concepts:

1. **JSON Schema Understanding**
   - Spark infers schema automatically
   - Two complex types: `array` and `struct`

2. **EXPLODE (Arrays → Rows)**
   ```python
   from pyspark.sql.functions import explode
   df.select(explode("array_column").alias("item"))
   ```
   - Converts array elements into separate rows
   - Use when: You have arrays and need each element as a row

3. **FLATTEN (Structs → Columns)**
   ```python
   df.select("struct_column.field1", "struct_column.field2")
   ```
   - Access nested fields with dot notation
   - Use when: You have nested objects and want flat columns

4. **Combine EXPLODE + FLATTEN**
   - For deeply nested JSON, chain operations:
     1. Explode outer array
     2. Flatten struct fields
     3. Explode inner arrays
     4. Flatten inner struct fields

### Common Pattern:
```python
df.select(explode("array1").alias("item")) \       # Explode array
  .select("item.field1",                      \       # Flatten struct
          "item.field2",                      \
          explode("item.nested_array").alias("nested")) \ # Explode nested
  .select("field1", "nested.nested_field")           # Flatten nested
```

### Next Steps:
- Practice with your own nested JSON files
- Try multiple levels of nesting
- Combine with SQL transformations